# Sub-clustering in Scanpy

In [ ]:
# This cell is labelled 'parameters' to work with papermill remotely
# Papermill will overwite the default local plate value below with whatever is passed to 
# the -p flag in the snakerule shell script
#os.system("conda activate eqtl_study") use this locally if using VScode
plate = 'plate1'
plate = globals().get("plate")
print(f"Processing plate: {plate}")

### Set Variables

In [ ]:
# Import custom utility packages, lists and functions
import sys
import os
if os.path.exists('/scratch/'):
    root_dir = '/scratch/c.c1477909/eQTL_study_2025/'
else:
    root_dir = '/Users/darren/Desktop/eQTL_study_2025/'
        
sys.path.append(root_dir + 'workflow/scripts/')

from init_env import *
from anndata_utils import *
from gene_lists import *

# Set variables
#resolutions = [0.1, 0.2, 0.3] # This takes a while per res; used in testing
var_genes = 6000
n_pcs = 50
batch_col = 'sample' 

### Load Data

In [ ]:

# Initialize the environment and get all paths and logger
logger, root_dir, sheets_dir, plate_path, scanpy_dir, report_dir = initialize_env(plate)
subcluster_dir = scanpy_dir + f'subclustering/' # maybe add to init_env.py later
os.makedirs(subcluster_dir, exist_ok=True)
logger.info("Loading data ...")
adata = sc.read(scanpy_dir + f'adata_clusters.h5ad')

### Set Level 1 - Cluster IDs

In [ ]:
logger.info("Setting cluster IDs ...")
# Set cluster names
cluster_anns = {
    
    '0': 'ExN-UL',
    '1': 'ExN-DL',
    '2': 'RG',
    '3': 'InN',
    '4': 'Endo-Peri',
    '5': 'OPC',
    '6': 'MG'
}

custom_palette = {
    'RG': '#FF5959',
    'ExN-UL': '#00B6EB',
    'InN': '#3CBB75FF',
    'ExN-DL': '#CEE5FD',
    'Endo-Peri': '#B200ED',
    'MG': '#F58231',
    'OPC': '#FDE725FF'
}

# Create a new column in adata.obs with cell type names
adata.obs['cell_type'] = adata.obs['leiden_harmony_0.1'].map(cluster_anns)

# Check the new column
print(adata.obs[['leiden_harmony_0.1', 'cell_type']].head())

### Restore raw counts for subclustering

In [ ]:
adata = adata.raw.to_adata() # Note that I saved the previous raw as the full data with all genes in clustering scripy: need to fix this
adata.layers["counts"] = adata.X.copy()
print(adata.layers)
print(adata.layers["counts"])

### Subset Scanpy object by cluster

In [ ]:
# Subset for each cell type
ul_exn_adata = adata[adata.obs['cell_type'] == 'ExN-UL'].copy()
dl_exn_adata = adata[adata.obs['cell_type'] == 'ExN-DL'].copy()
rg_adata = adata[adata.obs['cell_type'] == 'RG'].copy()
inn_adata = adata[adata.obs['cell_type'] == 'InN'].copy()

# Explicitly set .raw to preserve current (raw) .X for each subset before pipeline
ul_exn_adata.layers["counts"] = ul_exn_adata.X.copy()
dl_exn_adata.layers["counts"] = dl_exn_adata.X.copy()
rg_adata.layers["counts"] = rg_adata.X.copy()
inn_adata.layers["counts"] = inn_adata.X.copy()

# Remove main adata to free up memory
del adata

# Check the number of cells in each subset
print(ul_exn_adata.obs['cell_type'].value_counts())
print(dl_exn_adata.obs['cell_type'].value_counts())
print(rg_adata.obs['cell_type'].value_counts())
print(inn_adata.obs['cell_type'].value_counts())

### Run Scanpy pipeline on each subset

In [ ]:
# Define resolution per cell type
resolution_map = {
    'ExN-UL': 0.2,
    'ExN-DL': 0.2,
    'RG': 0.2,
    'InN': 0.1
}
# Process each subset
for subset_adata, cell_type in zip([ul_exn_adata, dl_exn_adata, rg_adata, inn_adata],
                                 ['ExN-UL', 'ExN-DL', 'RG', 'InN']):
    logger.info(f"Processing {cell_type} for subclustering ...")
    res = resolution_map[cell_type]

    # Ensure raw counts are available (if not, set adata.raw = adata before subsetting in earlier code)
    if 'raw' not in subset_adata.layers:
      subset_adata.raw = subset_adata.copy()  # Store raw for later if needed
    
    # Normalize and log transform
    sc.pp.normalize_total(subset_adata, target_sum=1e4)
    sc.pp.log1p(subset_adata)
    
    # Identify highly variable genes
    sc.pp.highly_variable_genes(subset_adata, n_top_genes = var_genes, flavor = "seurat_v3", batch_key = batch_col)
    
    # Scale the data
    sc.pp.scale(subset_adata, max_value=10)
    
    # Perform PCA
    sc.tl.pca(subset_adata, n_comps = n_pcs, use_highly_variable = True)
    
    # Compute the neighborhood graph
    sc.pp.neighbors(subset_adata, n_neighbors=10, n_pcs=n_pcs)
    
    # UMAP embedding
    sc.tl.umap(subset_adata)
    
    # Clustering with Leiden algorithm at specific resolution (pre-Harmony)
    cluster_key = f'leiden_{cell_type}_L2__{res}'
    sc.tl.leiden(subset_adata, resolution=res, key_added=cluster_key)

    # Batch correction with Harmony
    sce.pp.harmony_integrate(subset_adata, batch_col)
    subset_adata.obsm['X_pca'] = subset_adata.obsm['X_pca_harmony']
    sc.pp.neighbors(subset_adata, n_neighbors=10, n_pcs=n_pcs)
    sc.tl.umap(subset_adata) # Overwites pre-harmony X_umap

    # Clustering with Leiden algorithm at specific resolution (post-Harmony)
    cluster_key = f'leiden_{cell_type}_L2_harmony_{res}'
    sc.tl.leiden(subset_adata, resolution=res, key_added=cluster_key)



### UMAPs

In [ ]:
# Define distinct gene lists for each cell type
genes_per_type = {
    'ExN-UL': exN_genes + ["PLXNA4", "SATB2"],
    'ExN-DL': exN_genes + ["BCL11B", "TLE4"],
    'RG': r_glia_genes + ["GLI2"], 
    'InN': inN_genes + ["ERBB4"]
}
# List of subset AnnData objects and their corresponding cell types
subsets = [
    (ul_exn_adata, 'ExN-UL'),
    (dl_exn_adata, 'ExN-DL'),
    (rg_adata, 'RG'),
    (inn_adata, 'InN')
]

# Create a 2x4 plot for UMAPs (row 0) and violins (row 1)
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for idx, (subset_adata, cell_type) in enumerate(subsets):
    res = resolution_map[cell_type]
    
    # Add subcluster column with L1 prefix for this resolution
    cluster_key = f'leiden_{cell_type}_L2_harmony_{res}'
    subset_adata.obs['subcluster'] = subset_adata.obs[cluster_key].apply(lambda x: f'{cell_type}-{x}')
    
    # Plot UMAP for this cell type (row 0)
    ax_umap = axes[0, idx]
    sc.pl.umap(
        subset_adata,
        color='subcluster',
        ax=ax_umap,
        title=f'{cell_type} subcluster: Harmony L2 {res}',
        show=False
    )
    ax_umap.set_xlabel('UMAP1')
    ax_umap.set_ylabel('UMAP2')
    
    # Plot stacked violins for gene set grouped by subcluster (row 1)
    genes = genes_per_type[cell_type]
    valid_genes = [g for g in genes if g in subset_adata.var_names]
    ax_violin = axes[1, idx]
    sc.pl.stacked_violin(
        subset_adata,
        var_names=valid_genes,
        groupby='subcluster',
        ax=ax_violin,
        show=False
    )
    ax_violin.set_title(f'{cell_type} Gene Set Expression')

plt.tight_layout()
plt.show()

### Subcluster counts

In [ ]:
# Ensure all subclusters columns are added for each cell type's resolution
for subset_adata, cell_type in subsets:
    res = resolution_map[cell_type]
    cluster_key = f'leiden_{cell_type}_L2_harmony_{res}'
    subset_adata.obs[f'subcluster'] = subset_adata.obs[cluster_key].apply(lambda x: f'{cell_type}-{x}')

# Collect value counts into a list for DataFrame
data = []
for subset_adata, cell_type in subsets:
    res = resolution_map[cell_type]
    col = f'subcluster'
    counts = subset_adata.obs[col].value_counts()
    for subcluster, count in counts.items():
        data.append({
            'Resolution': res,
            'Cell_Type': cell_type,
            'Subcluster': subcluster,
            'Count': count
        })

# Create DataFrame and display
df = pd.DataFrame(data)
df

### Test number of nuceli per sample in each cell type

In [ ]:
# Group by sample and subcluster, count nuclei per group, unstack to wide format
logger.info("Check nuclei per sample per subcluster counts ...")

# Concat obs from all subsets (since main adata was deleted)
all_obs = pd.concat([subset_adata.obs for subset_adata, _ in subsets])

cell_sample_counts = (
    all_obs.groupby(['sample', 'subcluster']).size()
    .unstack(fill_value=0)
)

thresholds = [10, 20, 30]
for thresh in thresholds:
    print(f"\n--- Threshold: >= {thresh} nuclei per sample ---")
    results = {}
    for subcluster in cell_sample_counts.columns:
        num_qualifying_samples = (cell_sample_counts[subcluster] >= thresh).sum()
        results[subcluster] = num_qualifying_samples
        status = '✓ Meets criteria' if num_qualifying_samples >= 100 else '✗ Does not meet criteria'
        print(f"{subcluster}: {num_qualifying_samples} samples — {status}")
    
    # Optional: View as DataFrame
    print("Summary table:")
    print(pd.DataFrame.from_dict(results, orient='index', columns=[f'num_samples_>=_{thresh}']))

### Save

In [ ]:
# Save before generating pseudobulk data as we need raw counts for that
for subset_adata, cell_type in zip([ul_exn_adata, dl_exn_adata, rg_adata, inn_adata],
                                   ['ExN-UL', 'ExN-DL', 'RG', 'InN']):
   logger.info(f"Saving {cell_type} for h5ad file ...")
   subset_adata.write(subcluster_dir + f'adata_{cell_type}_subclusters.h5ad')
logger.info("All done.")

### Find overlapping samples with genotype data

In [ ]:
logger.info("Load genotypes sample list ...")
geno_sample_file = Path(root_dir + 'results/04GENOTYPES-POST/filtered/geno_sample_list.txt')
if not geno_sample_file.exists():
    print("Overlapping samples file does not exist")
else:
    print("Overlapping samples file already exists")
    with open(geno_sample_file, "r") as f:
        sample_set = set(f.read().splitlines())

### Generate Pseudobulk

In [ ]:
logger.info("Generating pseudobulk for subclusters ...")
pseudblk_dfs = {}

# Restore raw counts to .X for each subset before pseudobulk (matches your earlier full adata restoration)
for subset_adata, cell_type in subsets:
    logger.info(f"Restoring raw counts for {cell_type} ...")

    subset_adata.X = subset_adata.layers["counts"].copy()
    print(subset_adata.X)
    
for subset_adata, cell_type in subsets:
    print(f"Processing cell type: {cell_type}")
    
    # Loop through unique subclusters for the current cell type
    for subcluster in subset_adata.obs['subcluster'].unique():
        print(f"  Processing subcluster: {subcluster}")
        
        # Subset the data for the current subcluster (create a copy)
        sub_sub = subset_adata[subset_adata.obs['subcluster'] == subcluster].copy()
        
        # Initialize an empty list to store pseudobulk objects
        combined_pseudobulk_lst = []

        # Loop through unique samples
        for sample in sub_sub.obs['sample'].unique():

            # Remove samples not in the overlapping set
            if sample not in sample_set:  
                continue

            samp_sub = sub_sub[sub_sub.obs['sample'] == sample].copy()

            # Extract the raw counts
            counts_matrix = samp_sub.X
            
            # Ensure counts are stored as integers
            print(f"Max count value (should be integer): {counts_matrix.max()}")
            
            # Sum counts across cells in the sample
            pseudobulk_counts = counts_matrix.sum(axis=0).A1 if hasattr(counts_matrix, "A1") else counts_matrix.sum(axis=0)
            
            # Create an AnnData object with summed counts
            samp_adata = sc.AnnData(X=pseudobulk_counts.reshape(1, -1), var=samp_sub.var)
            samp_adata.obs_names = [sample]

            # Append to the list
            combined_pseudobulk_lst.append(samp_adata)

        # Only proceed if there are samples for this subcluster
        if combined_pseudobulk_lst:
            # Combine all samples into one AnnData object
            pseudobulk_adata = sc.concat(combined_pseudobulk_lst, join="outer")

            # Convert pseudobulk_adata to a DataFrame for the current subcluster
            pseudobulk_df = pd.DataFrame(pseudobulk_adata.X, columns=pseudobulk_adata.var_names, index=pseudobulk_adata.obs_names)
            print(f"pseudobulk_df dims: {pseudobulk_df.shape}")

            # Save the DataFrame to the dictionary
            pseudblk_dfs[subcluster] = pseudobulk_df

            # Save the pseudobulk data for this subcluster to a CSV file
            pseudobulk_df.to_csv(os.path.join(scanpy_dir, f'pseudobulk/{subcluster}_pseudobulk.csv'))

In [ ]:
for df in pseudblk_dfs:
    print(pseudblk_dfs[df].shape)